# Docker: воспроизводимое обучение модели

В практике собирается образ для конечной ML-задачи. При запуске контейнер обучает `RandomForestClassifier`, сохраняет модель и метрики во внешнее хранилище, затем завершает работу.

Результат практики:

- параметризованный `train_model.py`;
- `Dockerfile` с непривилегированным пользователем;
- небольшой build context через `.dockerignore`;
- сохранение артефактов через volume или bind mount;
- локальные тесты кода обучения.


## 1. Образ и контейнер

`docker build` читает Dockerfile и создаёт образ. Образ является неизменяемым шаблоном с Python, библиотеками, кодом и настройками запуска.

`docker run` создаёт контейнер из образа и запускает команду `CMD`. Контейнер имеет собственный writable layer. Этот слой не подходит для постоянного хранения модели, потому что удаляется вместе с контейнером.

```text
Dockerfile + build context = image
image + runtime settings = container
container + mounted storage = persistent model artifacts
```


## 2. Подготовка локального окружения

Откройте терминал в директории `practice_5` и установите зависимости:

```bash
python -m pip install -r requirements-dev.txt
```

Сначала проверяется сам Python-код. Это отделяет ошибку обучения от ошибки сборки или запуска Docker. В следующей ячейке артефакты создаются во временном каталоге `.notebook_artifacts`, который удаляется в конце notebook.


In [1]:
import json
import shutil
from pathlib import Path

import joblib

from train_model import TrainingConfig, train_model

artifact_dir = Path(".notebook_artifacts")
if artifact_dir.exists():
    shutil.rmtree(artifact_dir)

config = TrainingConfig(
    output_dir=artifact_dir,
    n_estimators=20,
    test_size=0.2,
    random_state=42,
    model_version="notebook-check",
)
metrics = train_model(config)


Model saved to .notebook_artifacts\iris_random_forest.joblib
Metrics saved to .notebook_artifacts\metrics.json
Accuracy: 0.9333


`TrainingConfig` хранит параметры одного запуска. В Docker те же значения читаются из `MODEL_DIR`, `MODEL_VERSION`, `N_ESTIMATORS`, `TEST_SIZE` и `RANDOM_STATE`.

`stratify=iris.target` сохраняет доли классов в train и test. Фиксированный `random_state` позволяет повторить разбиение и обучение. Значение `n_jobs=1` не позволяет одному учебному контейнеру занять все доступные CPU.


In [2]:
model_path = artifact_dir / "iris_random_forest.joblib"
metrics_path = artifact_dir / "metrics.json"
saved_metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
model = joblib.load(model_path)
prediction = int(model.predict([[5.1, 3.5, 1.4, 0.2]])[0])

print("model exists:", model_path.is_file())
print("metrics exists:", metrics_path.is_file())
print("accuracy:", round(saved_metrics["accuracy"], 4))
print("n_estimators:", saved_metrics["n_estimators"])
print("prediction for Iris sample:", prediction)


model exists: True
metrics exists: True
accuracy: 0.9333
n_estimators: 20
prediction for Iris sample: 0


## 3. Структура Dockerfile

Dockerfile выполняет четыре отдельные задачи:

1. выбирает базовый Python-образ;
2. устанавливает только runtime-зависимости;
3. копирует скрипт и создаёт каталог результатов;
4. переключается на пользователя `10001:10001` и задаёт команду обучения.

Обучение не выполняется при `docker build`. Сборка создаёт окружение, а обучение начинается при `docker run`.


In [3]:
dockerfile_text = Path("Dockerfile").read_text(encoding="utf-8")
print(dockerfile_text)


# syntax=docker/dockerfile:1

FROM python:3.12-slim

ARG BUILD_DATE=unknown
ARG VCS_REF=unknown

LABEL org.opencontainers.image.title="MIIT Iris trainer" \
      org.opencontainers.image.description="Reproducible Iris training job" \
      org.opencontainers.image.created="${BUILD_DATE}" \
      org.opencontainers.image.revision="${VCS_REF}"

ENV PYTHONDONTWRITEBYTECODE=1 \
    PYTHONUNBUFFERED=1 \
    MODEL_DIR=/app/models \
    MODEL_VERSION=1.0.0 \
    N_ESTIMATORS=100 \
    TEST_SIZE=0.3 \
    RANDOM_STATE=42

WORKDIR /app

COPY requirements.txt ./
RUN python -m pip install --no-cache-dir -r requirements.txt

RUN groupadd --gid 10001 appuser \
    && useradd --uid 10001 --gid 10001 --no-log-init --create-home appuser \
    && mkdir -p /app/models \
    && chown -R 10001:10001 /app

COPY --chown=10001:10001 train_model.py ./

USER 10001:10001

CMD ["python", "train_model.py"]



## 4. Build context и кэш

В команде `docker build -t miit-iris-trainer:1.0 .` последняя точка задаёт build context. `COPY` видит только файлы внутри этого каталога.

`.dockerignore` исключает Git metadata, виртуальные окружения, notebook, тесты и готовые модели. Это уменьшает контекст и риск попадания лишних файлов в сборку.

В Dockerfile сначала копируется `requirements.txt` и устанавливаются библиотеки, затем копируется `train_model.py`. Изменение кода не требует повторной установки зависимостей, если `requirements.txt` не изменился.


## 5. ARG и ENV

`ARG` доступен во время сборки. В практике через него передаются `BUILD_DATE` и `VCS_REF`, которые сохраняются в labels образа. `ARG`, объявленный до `FROM`, не становится автоматически доступен после `FROM`. Если значение нужно внутри этапа, `ARG` объявляется повторно после `FROM`.

`ENV` сохраняется в образе и доступен контейнеру. Значения обучения переопределяются при запуске через `-e`, например `-e N_ESTIMATORS=200`. Секреты нельзя хранить в `ARG` или `ENV` Dockerfile.


## 6. RUN, CMD и ENTRYPOINT

`RUN` выполняется при сборке и сохраняет результат в образе. `CMD` задаёт команду по умолчанию при запуске контейнера. `ENTRYPOINT` фиксирует основной исполняемый файл, а `CMD` может задавать его параметры по умолчанию.

Аргументы после имени образа заменяют `CMD`. `ENTRYPOINT` можно заменить флагом `--entrypoint`. В практике используется только `CMD ["python", "train_model.py"]`, поэтому команду легко заменить для диагностики.


## 7. Volume и bind mount

Контейнер запускается с `--rm`, поэтому после завершения удаляется. Модель должна сохраняться вне writable layer контейнера.

Именованный volume находится под управлением Docker:

```bash
docker volume create miit-iris-models
docker run --rm --mount type=volume,source=miit-iris-models,target=/app/models miit-iris-trainer:1.0
```

Bind mount связывает `/app/models` с выбранным каталогом хоста. В Dockerfile нет инструкции `VOLUME`: хранилище подключается явно при `docker run`, поэтому не создаётся неожиданный анонимный volume.


## 8. Сборка и запуск

Команды выполняются из `practice_5`:

```bash
docker build --build-arg BUILD_DATE=2026-11-10 --build-arg VCS_REF=local -t miit-iris-trainer:1.0 .
docker volume create miit-iris-models
docker run --rm --mount type=volume,source=miit-iris-models,target=/app/models -e N_ESTIMATORS=200 miit-iris-trainer:1.0
```

Проверка пользователя контейнера:

```bash
docker run --rm --entrypoint id miit-iris-trainer:1.0
```

Результат должен содержать `uid=10001` и не должен содержать `uid=0(root)`.


In [4]:
dockerignore_text = Path(".dockerignore").read_text(encoding="utf-8")

assert "FROM python:3.12-slim" in dockerfile_text
assert dockerfile_text.index("COPY requirements.txt") < dockerfile_text.index("COPY --chown=10001:10001 train_model.py")
assert 'USER 10001:10001' in dockerfile_text
assert 'CMD ["python", "train_model.py"]' in dockerfile_text
assert "python train_model.py" not in dockerfile_text.split("CMD")[0]
assert ".git" in dockerignore_text
assert "models" in dockerignore_text
assert "tests" in dockerignore_text

print("Dockerfile and .dockerignore checks passed.")


Dockerfile and .dockerignore checks passed.


## 9. Проверка

Локальные тесты запускаются командой `python -m pytest -q`. Они проверяют создание модели и метрик, загрузку модели, предсказание, переменные окружения и отказ при некорректных параметрах.

Полная Docker-проверка состоит из четырёх отдельных подтверждений:

1. `docker build` завершился успешно;
2. контейнер завершился с кодом `0`;
3. модель и метрики появились в подключённом хранилище;
4. процесс контейнера работал от UID `10001`.

Успешное локальное обучение Python не заменяет эти проверки Docker Engine.


In [5]:
assert model_path.is_file()
assert metrics_path.is_file()
assert saved_metrics == metrics
assert saved_metrics["n_estimators"] == 20
assert saved_metrics["model_version"] == "notebook-check"
assert prediction in {0, 1, 2}

shutil.rmtree(artifact_dir)
assert not artifact_dir.exists()

print("Notebook checks passed.")


Notebook checks passed.
